In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import numpy as np

# -----------------------------
# 1. Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 10
INPUT_SIZE = 28 * 28
HIDDEN1 = 256
HIDDEN2 = 128
NUM_CLASSES = 10
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts image to [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # Standard normalization for MNIST
])

# -----------------------------
# 4. Load MNIST dataset
# -----------------------------
train_dataset_full = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 54000
Validation samples: 6000
Test samples: 10000


In [4]:


class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10):
        super(MLP, self).__init__()
        
        self.model = nn.Sequential(
            nn.Flatten(),                       # 28x28 -> 784
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

model = MLP(INPUT_SIZE, HIDDEN1, HIDDEN2, NUM_CLASSES).to(device)
print(model)

# -----------------------------
# 6. Loss and optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

MLP(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [5]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", total_params)

Trainable parameters: 235146


In [6]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [7]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)


In [ ]:
import time

start_time = time.time()

best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_mlp_mnist.pth")

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_acc:.4f}")
end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

# 10. Test evaluation

model.load_state_dict(torch.load("best_mlp_mnist.pth", map_location=device))

test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro', zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

print("\nTest Results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("\nConfusion Matrix:")
print(cm)

Epoch [1/10]
  Train Loss: 0.3310 | Train Acc: 0.8996
  Val Loss:   0.1417 | Val Acc:   0.9568
Epoch [2/10]
  Train Loss: 0.1423 | Train Acc: 0.9571
  Val Loss:   0.1019 | Val Acc:   0.9683
Epoch [3/10]
  Train Loss: 0.1023 | Train Acc: 0.9683
  Val Loss:   0.0903 | Val Acc:   0.9720
Epoch [4/10]
  Train Loss: 0.0822 | Train Acc: 0.9742
  Val Loss:   0.0782 | Val Acc:   0.9777
Epoch [5/10]
  Train Loss: 0.0722 | Train Acc: 0.9771
  Val Loss:   0.0781 | Val Acc:   0.9797
Epoch [6/10]
  Train Loss: 0.0604 | Train Acc: 0.9808
  Val Loss:   0.0776 | Val Acc:   0.9762
Epoch [7/10]
  Train Loss: 0.0558 | Train Acc: 0.9824
  Val Loss:   0.0789 | Val Acc:   0.9760
Epoch [8/10]
  Train Loss: 0.0505 | Train Acc: 0.9834
  Val Loss:   0.0815 | Val Acc:   0.9777
Epoch [9/10]
  Train Loss: 0.0443 | Train Acc: 0.9849
  Val Loss:   0.0789 | Val Acc:   0.9780
Epoch [10/10]
  Train Loss: 0.0406 | Train Acc: 0.9869
  Val Loss:   0.0712 | Val Acc:   0.9802

Training complete.
Best validation accuracy: 0.9